In [1]:
# ==============================================================
# 11_classification_stacking.ipynb
# --------------------------------------------------------------
# Meta-learning (Stacking) ensemble to blend multiple classifiers
# ==============================================================

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings("ignore")

# ==============================================================
# Configuration
# ==============================================================
project_root = Path("C:/JupyterProjects/Stock_ML_Project")
data_dir = project_root / "Data" / "Processed" / "enhanced"
results_dir = project_root / "Results"
results_dir.mkdir(parents=True, exist_ok=True)

tickers = {
    "RELIANCE": data_dir / "reliance_enhanced_model_ready.csv",
    "TCS": data_dir / "tcs_enhanced_model_ready.csv",
    "HDFCBANK": data_dir / "hdfcbank_enhanced_model_ready.csv",
}

# ==============================================================
# Helper
# ==============================================================
def evaluate(y_true, y_pred, y_proba):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_proba)
    }

# ==============================================================
# Main
# ==============================================================
results = []

for ticker, path in tickers.items():
    print(f"\n=== Processing {ticker} ===")
    if not path.exists():
        print(f"⚠️ Missing file: {path}")
        continue

    df = pd.read_csv(path)
    print(f"  Loaded dataset: {df.shape}")

    target_col = "Target_Cls"
    if target_col not in df.columns:
        print(f"⚠️ Skipping {ticker} — Target_Cls not found.")
        continue

    df = df.replace([np.inf, -np.inf], np.nan)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    df_numeric = df[numeric_cols].dropna(axis=1, how="all")

    imputer = SimpleImputer(strategy="mean")
    df_numeric[df_numeric.columns] = imputer.fit_transform(df_numeric)

    y = df_numeric[target_col]
    X = df_numeric.drop(columns=[target_col, "Target_Reg"], errors="ignore")

    X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False, test_size=0.2)

    base_models = [
        ("logistic", LogisticRegression(max_iter=1000)),
        ("tree", DecisionTreeClassifier(random_state=42)),
        ("rf", RandomForestClassifier(n_estimators=150, random_state=42)),
        ("svm", SVC(kernel='rbf', probability=True))
    ]

    meta_model = LogisticRegression(max_iter=1000)

    stacking_clf = StackingClassifier(
        estimators=base_models,
        final_estimator=meta_model,
        cv=5,
        stack_method="predict_proba"
    )

    stacking_clf.fit(X_train, y_train)
    y_proba = stacking_clf.predict_proba(X_test)[:, 1]
    y_pred = (y_proba > 0.5).astype(int)

    metrics = evaluate(y_test, y_pred, y_proba)
    metrics.update({"Ticker": ticker, "Model": "StackingClassifier"})
    results.append(metrics)

    print(f"  ✅ StackingClassifier: Acc={metrics['Accuracy']:.3f}, F1={metrics['F1']:.3f}, Prec={metrics['Precision']:.3f}, Rec={metrics['Recall']:.3f}, AUC={metrics['ROC_AUC']:.3f}")

# ==============================================================
# Save Results
# ==============================================================
results_df = pd.DataFrame(results)
save_path = results_dir / "classification_stacking_results.csv"
results_df.to_csv(save_path, index=False)

print(f"\n✅ Classification Stacking completed. Results saved to: {save_path}")
display(results_df)



=== Processing RELIANCE ===
  Loaded dataset: (1460, 25)
  ✅ StackingClassifier: Acc=0.483, F1=0.604, Prec=0.509, Rec=0.742, AUC=0.480

=== Processing TCS ===
  Loaded dataset: (1460, 25)
  ✅ StackingClassifier: Acc=0.490, F1=0.656, Prec=0.491, Rec=0.986, AUC=0.511

=== Processing HDFCBANK ===
  Loaded dataset: (1460, 25)
  ✅ StackingClassifier: Acc=0.524, F1=0.676, Prec=0.531, Rec=0.929, AUC=0.478

✅ Classification Stacking completed. Results saved to: C:\JupyterProjects\Stock_ML_Project\Results\classification_stacking_results.csv


,Accuracy,Precision,Recall,F1,ROC_AUC,Ticker,Model
0,0.482877,0.508850,0.741935,0.603675,0.480198,RELIANCE,StackingClassifier
1,0.489726,0.491349,0.986111,0.655889,0.510557,TCS,StackingClassifier
2,0.523973,0.531136,0.929487,0.675991,0.477564,HDFCBANK,StackingClassifier
